In [3]:
# ============================
# 📦 Essential Libraries
# ============================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns


In [4]:
# ============================================
# 📂 Load all raw CSV datasets (collaborative-safe)
# ============================================

import os
import pandas as pd

# 1) Locate project root by searching for "data" folder
BASE_DIR = os.getcwd()
while "data" not in os.listdir(BASE_DIR) and os.path.dirname(BASE_DIR) != BASE_DIR:
    BASE_DIR = os.path.dirname(BASE_DIR)

RAW_DIR = os.path.join(BASE_DIR, "data", "raw")

print("Project root:", BASE_DIR)
print("Raw data folder:", RAW_DIR)
print("Files in raw:", os.listdir(RAW_DIR))
print("=====================================\n")

# 2) Load each dataset into a named DataFrame

df_cpi = pd.read_csv(os.path.join(RAW_DIR, "ons_cpi.csv"), low_memory=False)
df_interest = pd.read_csv(os.path.join(RAW_DIR, "boe_interest.csv"))
df_exchange = pd.read_csv(os.path.join(RAW_DIR, "exchange_rates.csv"))
df_gdp = pd.read_csv(os.path.join(RAW_DIR, "GDP_growth_Rate.csv"))
df_unemp = pd.read_csv(os.path.join(RAW_DIR, "Unemployment_Rate.csv"))
df_oil = pd.read_csv(os.path.join(RAW_DIR, "petrol_oil_everage_price_change.csv"))

print("Loaded DataFrames:")
print("  df_cpi      -> ons_cpi.csv")
print("  df_interest -> boe_interest.csv")
print("  df_exchange -> exchange_rates.csv")
print("  df_gdp      -> GDP_growth_Rate.csv")
print("  df_unemp    -> Unemployment_Rate.csv")
print("  df_oil      -> petrol_oil_everage_price_change.csv")


Project root: c:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI
Raw data folder: c:\Users\reset\OneDrive - UWE Bristol\Master_project\UK-inflation-forecasting-XAI\data\raw
Files in raw: ['.Rhistory', 'boe_interest.csv', 'exchange_rates.csv', 'GDP_growth_Rate.csv', 'ons_cpi.csv', 'petrol_oil_everage_price_change.csv', 'Unemployment_Rate.csv']

Loaded DataFrames:
  df_cpi      -> ons_cpi.csv
  df_interest -> boe_interest.csv
  df_exchange -> exchange_rates.csv
  df_gdp      -> GDP_growth_Rate.csv
  df_unemp    -> Unemployment_Rate.csv
  df_oil      -> petrol_oil_everage_price_change.csv


## 🎯 CPI Column Selection  
This step extracts the two key variables required from the CPI dataset:

- **Title** — the date or period identifier  
- **CPI ANNUAL RATE 00: ALL ITEMS 2015=100** — the annual inflation rate (main target variable)

A separate working DataFrame (`cpi_clean`) is created to allow further cleaning and transformation without modifying the original dataset.


In [5]:
# Extract two important CPI columns
cpi_clean = df_cpi[["Title", "CPI ANNUAL RATE 00: ALL ITEMS 2015=100"]].copy()
cpi_clean.head(10000)


,Title,CPI ANNUAL RATE 00: ALL ITEMS 2015=100
0,CDID,D7G7
1,PreUnit,NaN
2,Unit,%
3,Release Date,22-10-2025
4,Next release,19 November 2025
...,...,...
1476,2025 MAY,3.4
1477,2025 JUN,3.6
1478,2025 JUL,3.8
1479,2025 AUG,3.8


In [6]:
cpi_clean.dtypes


Title                                     object
CPI ANNUAL RATE 00: ALL ITEMS 2015=100    object
dtype: object

## 🧹 CPI Data Cleaning  
This step performs the initial cleaning of the CPI dataset:

- Converts `Title` to a proper datetime format  
- Converts the inflation variable to numeric values  
- Renames `Title` to `Date` to standardise the time index  

This prepares the CPI data for later merging with other economic indicators.


In [7]:
# Month patterns
months = "JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC"

# Keep only rows that contain a month abbreviation
cpi_monthly = cpi_clean[cpi_clean["Title"].str.contains(months, na=False)].copy()

# Drop fully empty rows
cpi_monthly = cpi_monthly.dropna(how="all")

# Drop rows where Title or CPI value is missing
cpi_monthly = cpi_monthly.dropna(subset=["Title", "CPI ANNUAL RATE 00: ALL ITEMS 2015=100"])

# Show first few cleaned rows
cpi_monthly.head(10)


,Title,CPI ANNUAL RATE 00: ALL ITEMS 2015=100
1040,1989 JAN,4.9
1041,1989 FEB,5.0
1042,1989 MAR,5.0
1043,1989 APR,5.3
1044,1989 MAY,5.3
1045,1989 JUN,5.2
1046,1989 JUL,5.2
1047,1989 AUG,5.0
1048,1989 SEP,5.2
1049,1989 OCT,5.5


In [8]:
cpi_monthly["Date"] = pd.to_datetime(cpi_monthly["Title"], format="%Y %b", errors="coerce")


In [12]:
cpi_monthly["CPI ANNUAL RATE 00: ALL ITEMS 2015=100"] = pd.to_numeric(
    cpi_monthly["CPI ANNUAL RATE 00: ALL ITEMS 2015=100"],
    errors="coerce"
)


In [9]:
cpi_monthly.head(100)

,Title,CPI ANNUAL RATE 00: ALL ITEMS 2015=100,Date
1040,1989 JAN,4.9,1989-01-01
1041,1989 FEB,5.0,1989-02-01
1042,1989 MAR,5.0,1989-03-01
1043,1989 APR,5.3,1989-04-01
1044,1989 MAY,5.3,1989-05-01
...,...,...,...
1135,1996 DEC,2.3,1996-12-01
1136,1997 JAN,2.1,1997-01-01
1137,1997 FEB,1.9,1997-02-01
1138,1997 MAR,1.7,1997-03-01


## 🧹 Removing Redundant Columns  
After converting the `Title` field into a proper datetime variable (`Date`),  
the original `Title` column becomes unnecessary.  
This step removes it, leaving only the cleaned `Date` column and the CPI annual inflation rate.


In [14]:
cpi_monthly = cpi_monthly.drop(columns=["Title"])
cpi_monthly.head()


,CPI ANNUAL RATE 00: ALL ITEMS 2015=100,Date
1040,4.9,1989-01-01
1041,5.0,1989-02-01
1042,5.0,1989-03-01
1043,5.3,1989-04-01
1044,5.3,1989-05-01


## 🎯 GDP Column Extraction  
This step selects the date column (`Title`) and the key GDP indicator  
**“Gross Value Added – Monthly (3 month on 3 month growth) : CVM SA”**.  
The resulting DataFrame (`gdp_clean`) will be cleaned and prepared for merging with other monthly datasets.


In [10]:
gdp_clean = df_gdp[[
    "Title",
    "Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA"
]].copy()


In [11]:
gdp_clean.dtypes
gdp_clean.head(100)



,Title,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA
0,CDID,ED3H
1,PreUnit,NaN
2,Unit,NaN
3,Release Date,16-10-2025
4,Next release,13 November 2025
...,...,...
95,2004 JUN,0.4
96,2004 JUL,0.5
97,2004 AUG,0.4
98,2004 SEP,0.3


## 🧹 GDP 3-Month Growth Cleaning  
This step filters the GDP dataset to monthly observations, converts the `Title` field into a proper datetime variable (`Date`),  
and converts the selected GDP growth series  
**"Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA"**  
to a numeric variable named `GDP_3m3m_Growth`.  
The cleaned data is sorted by date and stored in `gdp_monthly` for later merging.


In [12]:
# Keep only rows that contain a month abbreviation in Title
months = "JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC"

gdp_monthly = gdp_clean[gdp_clean["Title"].str.contains(months, na=False)].copy()

# Drop fully empty rows
gdp_monthly = gdp_monthly.dropna(how="all")

# Drop rows where Title or GDP value is missing
gdp_col = "Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA"
gdp_monthly = gdp_monthly.dropna(subset=["Title", gdp_col])

gdp_monthly.head(10)


,Title,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA
11,1997 JUN,1.0
12,1997 JUL,0.6
13,1997 AUG,0.9
14,1997 SEP,0.8
15,1997 OCT,1.0
16,1997 NOV,1.0
17,1997 DEC,1.4
18,1998 JAN,1.4
19,1998 FEB,1.4
20,1998 MAR,0.8


In [13]:
# Convert Title to datetime (same pattern as CPI, e.g. "1989 JAN")
gdp_monthly["Date"] = pd.to_datetime(gdp_monthly["Title"], format="%Y %b", errors="coerce")

# Convert GDP growth column to numeric
gdp_monthly[gdp_col] = pd.to_numeric(gdp_monthly[gdp_col], errors="coerce")

# Drop old Title column
gdp_monthly = gdp_monthly.drop(columns=["Title"])


# Sort by date and reset index
gdp_monthly = gdp_monthly.sort_values("Date").reset_index(drop=True)

gdp_monthly.head()


,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA,Date
0,1.0,1997-06-01
1,0.6,1997-07-01
2,0.9,1997-08-01
3,0.8,1997-09-01
4,1.0,1997-10-01


## 🏦 Interest Rate Dataset Overview  
This step provides an initial inspection of the interest rate dataset.  
The output includes:

- Total number of rows and columns  
- First ten observations for visual inspection  
- A complete list of column names  

This overview is used to identify the date field and the primary Bank Rate series before cleaning and selection.


In [14]:
print("Shape:", df_interest.shape)
print("=====================================\n")

print("First 10 rows:")
display(df_interest.head(1000))
print("=====================================\n")

print("Column names:")
for col in df_interest.columns:
    print("-", col)


Shape: (2526, 2)

First 10 rows:


,Date,Bank Rate
0,27-11-2015,0.50
1,30-11-2015,0.50
2,01-12-2015,0.50
3,02-12-2015,0.50
4,03-12-2015,0.50
...,...,...
995,05-11-2019,0.75
996,06-11-2019,0.75
997,07-11-2019,0.75
998,08-11-2019,0.75



Column names:
- Date
- Bank Rate


## 🏦 Interest Rate Monthly Aggregation  
The daily Bank Rate series is converted into a monthly series by:

1. Parsing the `Date` column as a proper datetime (DD–MM–YYYY).  
2. Ensuring `Bank Rate` is stored as a numeric variable.  
3. Creating a `YearMonth` key and, after sorting by date,  
   selecting the last observation in each month to represent the monthly Bank Rate.  

The resulting cleaned series is stored in `interest_monthly`.


In [16]:
# ============================================
# 🏦 Clean interest rate data to monthly series
# ============================================

# 1) Convert Date to datetime (day-first format: DD-MM-YYYY)
df_interest["Date"] = pd.to_datetime(df_interest["Date"], dayfirst=True, errors="coerce")

# 2) Ensure Bank Rate is numeric
df_interest["Bank Rate"] = pd.to_numeric(df_interest["Bank Rate"], errors="coerce")

# 3) Create Year-Month key
df_interest["YearMonth"] = df_interest["Date"].dt.to_period("M")

# 4) Sort by date so "last in month" really means latest date
df_interest = df_interest.sort_values("Date")

# 5) Keep the latest observation in each month
interest_monthly = df_interest.groupby("YearMonth").tail(1).copy()

# 6) Drop helper column and tidy up
interest_monthly = interest_monthly.drop(columns=["YearMonth"])
interest_monthly = interest_monthly.sort_values("Date").reset_index(drop=True)

# 7) Quick check
interest_monthly.head(1000)


,Date,Bank Rate
0,2015-11-30,0.50
1,2015-12-31,0.50
2,2016-01-29,0.50
3,2016-02-29,0.50
4,2016-03-31,0.50
...,...,...
116,2025-07-31,4.25
117,2025-08-29,4.00
118,2025-09-30,4.00
119,2025-10-31,4.00
